# Extração Copa 2026 — modo repositório incremental, sem API key

Este notebook roda **dentro do repositório**, lê os arquivos em `data/database/` quando existirem, busca dados somente em fontes públicas sem chave e salva os resultados novamente no repositório.

O modo incremental preserva informações já encontradas e **preenche apenas campos vazios/NA** quando uma nova execução encontrar dados novos. Não usa proxy, não estima dados e não sobrescreve dados bons por `NA`.

Saída padrão:

`data/database/enriched_jogadores_tecnicos/`

In [ ]:
import sys, subprocess, importlib.util
def ensure_package(package, import_name=None):
    import_name = import_name or package
    if importlib.util.find_spec(import_name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])
    print(package, "ok")

for pkg, imp in [("pandas","pandas"), ("requests","requests"), ("tqdm","tqdm")]:
    ensure_package(pkg, imp)

In [ ]:
from pathlib import Path
import os, re, json, time, random, zipfile, hashlib
import datetime as dt
from urllib.parse import quote

import pandas as pd
import requests
from tqdm.auto import tqdm

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

NA = "NA"

def now_iso():
    return dt.datetime.now(dt.timezone.utc).replace(microsecond=0).isoformat()

# ────────────────────────────────────────────
# CONFIGURAÇÃO DO MODO REPOSITÓRIO
# ────────────────────────────────────────────

UPDATE_ONLY_MISSING = True
SAVE_INSIDE_REPOSITORY = True

# Se quiser apontar manualmente para um ZIP específico:
MANUAL_ZIP_PATH = None

REQUEST_DELAY_SECONDS = (2.5, 6.0)
REQUEST_TIMEOUT = 30
PLAYER_LIMIT = None

USE_WIKIDATA = True
USE_WIKIPEDIA_SUMMARY = True
USE_FIFA_SQUAD_PDF = False
USE_STATSBOMB_OPEN_DATA = False

FIFA_SQUAD_PDF_URL = "https://fdp.fifa.org/assetspublic/ce281/pdf/SquadLists-English.pdf"
STATSBOMB_BASE = "https://raw.githubusercontent.com/statsbomb/open-data/master/data"

HEADERS = {"User-Agent": "copa-2026-dataset-research/1.0 (no api key; polite requests)"}

def find_repo_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for p in [start] + list(start.parents):
        if (p / ".git").exists() or (p / "data" / "database").exists():
            return p
    return start

REPO_ROOT = find_repo_root()
REPO_DATA_DIR = REPO_ROOT / "data" / "database"
OUTPUT_DIR = REPO_ROOT / "data" / "database" / "enriched_jogadores_tecnicos"
INPUT_DIR = REPO_ROOT / ".copa_2026_input_unzipped"
CACHE_DIR = REPO_ROOT / ".copa_2026_request_cache"

for p in [OUTPUT_DIR, INPUT_DIR, CACHE_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("REPO_ROOT:", REPO_ROOT)
print("REPO_DATA_DIR:", REPO_DATA_DIR if REPO_DATA_DIR.exists() else "não encontrado")
print("OUTPUT_DIR:", OUTPUT_DIR)
print("UPDATE_ONLY_MISSING:", UPDATE_ONLY_MISSING)

In [ ]:
# ────────────────────────────────────────────
# HTTP com delay + cache + log
# ────────────────────────────────────────────

request_log = []
sources_rows = []
source_counter = 0

def safe_filename_from_url(url: str) -> str:
    return hashlib.sha256(url.encode("utf-8")).hexdigest()

def cache_path_for(url, suffix="json"):
    return CACHE_DIR / f"{safe_filename_from_url(url)}.{suffix}"

def add_source(category, entity_type, entity_name, url, publisher, title="", date_published=NA,
               data_collected="", reliability_0_100=70, notes=""):
    global source_counter
    source_counter += 1
    sid = f"SRC{source_counter:05d}"
    sources_rows.append({
        "source_id": sid,
        "category": category,
        "entity_type": entity_type,
        "entity_name": entity_name,
        "url": url,
        "publisher": publisher,
        "title": title,
        "date_published": date_published,
        "date_accessed": now_iso(),
        "data_collected": data_collected,
        "reliability_0_100": reliability_0_100,
        "notes": notes,
    })
    return sid

def polite_get(url, *, as_json=False, as_bytes=False, use_cache=True, publisher="unknown", retries=2):
    suffix = "json" if as_json else "bin" if as_bytes else "txt"
    cp = cache_path_for(url, suffix=suffix)
    if use_cache and cp.exists():
        request_log.append({"url": url, "status": "cache_hit", "timestamp": now_iso()})
        if as_json:
            return json.loads(cp.read_text(encoding="utf-8"))
        if as_bytes:
            return cp.read_bytes()
        return cp.read_text(encoding="utf-8", errors="replace")

    sleep_s = random.uniform(*REQUEST_DELAY_SECONDS)
    print(f"Aguardando {sleep_s:.1f}s antes da requisição: {url[:120]}")
    time.sleep(sleep_s)

    last_err = None
    for attempt in range(retries + 1):
        try:
            r = requests.get(url, headers=HEADERS, timeout=REQUEST_TIMEOUT)
            request_log.append({"url": url, "status_code": r.status_code, "attempt": attempt+1, "timestamp": now_iso(), "publisher": publisher})
            if r.status_code == 429 and attempt < retries:
                time.sleep(15 * (attempt + 1))
                continue
            r.raise_for_status()
            if as_json:
                data = r.json()
                cp.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")
                return data
            if as_bytes:
                cp.write_bytes(r.content)
                return r.content
            cp.write_text(r.text, encoding="utf-8")
            return r.text
        except Exception as e:
            last_err = e
            request_log.append({"url": url, "status": "error", "error": repr(e), "attempt": attempt+1, "timestamp": now_iso()})
            if attempt < retries:
                time.sleep(5 * (attempt + 1))
    print("Falha:", url, last_err)
    return None

In [ ]:
# ────────────────────────────────────────────
# Localização de arquivos no repositório/ZIP
# ────────────────────────────────────────────

def repo_find_file(filename):
    filename = filename.lower()
    search_dirs = [REPO_DATA_DIR, REPO_ROOT, REPO_ROOT / "data", REPO_ROOT / "datasets", REPO_ROOT / "notebooks", Path.cwd(), Path("/mnt/data"), Path("/content")]
    found = []
    for base in search_dirs:
        if base.exists():
            try:
                found.extend([p for p in base.rglob("*") if p.is_file() and p.name.lower() == filename])
            except Exception:
                pass
    found = sorted(set(found), key=lambda p: (0 if str(p).startswith(str(REPO_DATA_DIR)) else 1, len(str(p))))
    return found[0] if found else None

def repo_find_zip():
    if MANUAL_ZIP_PATH is not None:
        p = Path(MANUAL_ZIP_PATH).expanduser()
        if p.exists():
            return p
    patterns = ["pacote_dataset_copa_2026_features*.zip", "*copa*2026*features*.zip", "*.zip"]
    search_dirs = [REPO_ROOT, Path.cwd(), Path("/mnt/data"), Path("/content")]
    found = []
    for base in search_dirs:
        if base.exists():
            for pat in patterns:
                try:
                    found.extend(base.rglob(pat))
                except Exception:
                    pass
    found = sorted(set([p for p in found if p.is_file() and p.suffix.lower() == ".zip"]),
                   key=lambda p: (0 if "pacote_dataset_copa_2026_features" in p.name else 1, -p.stat().st_mtime))
    return found[0] if found else None

def read_csv_flexible(path):
    if path is None:
        return None
    try:
        return pd.read_csv(path, dtype=str, keep_default_na=False, na_values=[])
    except UnicodeDecodeError:
        return pd.read_csv(path, dtype=str, keep_default_na=False, na_values=[], encoding="latin1")

def find_file_by_name_in_input(name):
    matches = [p for p in INPUT_DIR.rglob("*") if p.is_file() and p.name.lower() == name.lower()]
    return matches[0] if matches else None

required_candidates = {
    "players_database": repo_find_file("players_database.csv"),
    "team_strengths": repo_find_file("team_strengths.csv"),
    "teams_tactical": repo_find_file("teams_tactical.csv"),
    "matches": repo_find_file("matches.csv"),
    "dataset_features": repo_find_file("dataset_copa_2026_features_pesquisadas.csv"),
    "data_dictionary_existing": repo_find_file("data_dictionary_dataset_copa_2026.csv"),
    "sources_existing": repo_find_file("sources_used.csv"),
}

if required_candidates["dataset_features"] is None:
    zp = repo_find_zip()
    if zp is None:
        try:
            from google.colab import files
            print("Nenhum ZIP encontrado. Envie o pacote ZIP...")
            uploaded = files.upload()
            if uploaded:
                zp = Path(next(iter(uploaded.keys()))).resolve()
        except Exception:
            pass
    if zp is None:
        raise FileNotFoundError("Não encontrei o dataset nem o ZIP do pacote. Coloque os CSVs ou o ZIP dentro do repositório.")
    print("ZIP selecionado:", zp)
    for old in sorted(INPUT_DIR.rglob("*"), reverse=True):
        if old.is_file():
            old.unlink()
        elif old.is_dir():
            try:
                old.rmdir()
            except OSError:
                pass
    with zipfile.ZipFile(zp, "r") as z:
        z.extractall(INPUT_DIR)
    fallback = {
        "players_database": find_file_by_name_in_input("players_database.csv"),
        "team_strengths": find_file_by_name_in_input("team_strengths.csv"),
        "teams_tactical": find_file_by_name_in_input("teams_tactical.csv"),
        "matches": find_file_by_name_in_input("matches.csv"),
        "dataset_features": find_file_by_name_in_input("dataset_copa_2026_features_pesquisadas.csv"),
        "data_dictionary_existing": find_file_by_name_in_input("data_dictionary_dataset_copa_2026.csv"),
        "sources_existing": find_file_by_name_in_input("sources_used.csv"),
    }
    for k, v in fallback.items():
        if required_candidates[k] is None and v is not None:
            required_candidates[k] = v

print("Arquivos usados como entrada:")
for k, v in required_candidates.items():
    print(f"{k}: {v if v else 'NÃO ENCONTRADO'}")

players_db = read_csv_flexible(required_candidates["players_database"])
team_strengths = read_csv_flexible(required_candidates["team_strengths"])
teams_tactical = read_csv_flexible(required_candidates["teams_tactical"])
matches_df = read_csv_flexible(required_candidates["matches"])
dataset_features = read_csv_flexible(required_candidates["dataset_features"])
existing_sources = read_csv_flexible(required_candidates["sources_existing"])

for name, df in [("players_database", players_db), ("team_strengths", team_strengths), ("teams_tactical", teams_tactical), ("matches", matches_df), ("dataset_features", dataset_features), ("existing_sources", existing_sources)]:
    print(name, "=>", None if df is None else df.shape)

In [ ]:
# ────────────────────────────────────────────
# Utilitários de normalização
# ────────────────────────────────────────────

def normalize_colname(c):
    c = str(c).strip().lower()
    c = re.sub(r"[áàãâä]", "a", c)
    c = re.sub(r"[éèêë]", "e", c)
    c = re.sub(r"[íìîï]", "i", c)
    c = re.sub(r"[óòõôö]", "o", c)
    c = re.sub(r"[úùûü]", "u", c)
    c = re.sub(r"[ç]", "c", c)
    c = re.sub(r"[^a-z0-9]+", "_", c)
    return re.sub(r"_+", "_", c).strip("_")

def standardize_players_df(df):
    if df is None:
        return pd.DataFrame(columns=["player_id", "selecao", "jogador", "posicao", "clube", "pais_do_clube", "liga", "idade", "caps_selecao", "gols_selecao"])
    out = df.copy()
    out.columns = [normalize_colname(c) for c in out.columns]
    aliases = {"nome":"jogador","player":"jogador","player_name":"jogador","name":"jogador","team":"selecao","equipe":"selecao","country":"selecao","national_team":"selecao","position":"posicao","pos":"posicao","club":"clube","age":"idade","caps":"caps_selecao","goals":"gols_selecao","international_goals":"gols_selecao"}
    for old, new in aliases.items():
        if old in out.columns and new not in out.columns:
            out[new] = out[old]
    required = ["player_id", "selecao", "jogador", "posicao", "clube", "pais_do_clube", "liga", "idade", "caps_selecao", "gols_selecao"]
    for c in required:
        if c not in out.columns:
            out[c] = NA
    if len(out) and ((out["player_id"] == NA).all() or out["player_id"].eq("").all()):
        out["player_id"] = [f"P{i+1:05d}" for i in range(len(out))]
    return out[required].replace({"": NA})

players_base = standardize_players_df(players_db)
if PLAYER_LIMIT is not None and len(players_base) > PLAYER_LIMIT:
    players_base = players_base.head(PLAYER_LIMIT).copy()

def get_teams_from_inputs():
    teams = []
    for df in [dataset_features, matches_df]:
        if df is not None:
            tmp = df.copy()
            tmp.columns = [normalize_colname(c) for c in tmp.columns]
            for c in ["equipe1", "equipe2", "team1", "team2"]:
                if c in tmp.columns:
                    teams += list(tmp[c].dropna().astype(str))
    if "selecao" in players_base.columns:
        teams += list(players_base["selecao"].dropna().astype(str))
    return sorted({t for t in teams if t and t != NA})

teams = get_teams_from_inputs()
print("Jogadores:", len(players_base))
print("Seleções:", len(teams))
display(players_base.head())

In [ ]:
# ────────────────────────────────────────────
# Coletores sem key: Wikidata/Wikipedia
# ────────────────────────────────────────────

WIKIDATA_API = "https://www.wikidata.org/w/api.php"
WIKIDATA_ENTITY = "https://www.wikidata.org/wiki/Special:EntityData/{qid}.json"
WIKIPEDIA_SEARCH = "https://en.wikipedia.org/w/api.php"
WIKIPEDIA_SUMMARY = "https://en.wikipedia.org/api/rest_v1/page/summary/{title}"

def wikidata_search_entity(search, language="en", limit=5):
    url = f"{WIKIDATA_API}?action=wbsearchentities&search={quote(search)}&language={language}&format=json&limit={limit}"
    data = polite_get(url, as_json=True, publisher="Wikidata")
    return data.get("search", []) if data else []

def wikidata_get_entity(qid):
    data = polite_get(WIKIDATA_ENTITY.format(qid=qid), as_json=True, publisher="Wikidata")
    return data.get("entities", {}).get(qid) if data else None

def wd_claim_values(entity, prop):
    return [c.get("mainsnak", {}).get("datavalue", {}).get("value") for c in entity.get("claims", {}).get(prop, [])]

def wd_entity_id_from_value(value):
    if isinstance(value, dict) and "id" in value:
        return value["id"]
    if isinstance(value, dict) and "numeric-id" in value:
        return "Q" + str(value["numeric-id"])
    return None

def wd_label_from_entity_id(qid, lang_priority=("pt","en","es","fr")):
    ent = wikidata_get_entity(qid)
    if not ent:
        return NA
    labels = ent.get("labels", {})
    for lang in lang_priority:
        if lang in labels:
            return labels[lang]["value"]
    return next(iter(labels.values()))["value"] if labels else NA

def wikidata_enrich_person_by_name(name, context="", entity_type="player"):
    if not USE_WIKIDATA or not name or name == NA:
        return {}
    candidates = wikidata_search_entity(f"{name} {context}".strip(), language="en", limit=5) or wikidata_search_entity(name, language="en", limit=5)
    if not candidates:
        return {"source_id": NA, "notes": "Wikidata entity not found"}
    qid = candidates[0]["id"]
    ent = wikidata_get_entity(qid)
    if not ent:
        return {"source_id": NA, "notes": "Wikidata entity fetch failed"}
    countries, positions, clubs = [], [], []
    for prop, target in [("P27", countries), ("P413", positions), ("P54", clubs)]:
        for val in wd_claim_values(ent, prop):
            eid = wd_entity_id_from_value(val)
            if eid:
                target.append(wd_label_from_entity_id(eid))
    sid = add_source("player_stats" if entity_type=="player" else "coach_stats", entity_type, name, WIKIDATA_ENTITY.format(qid=qid), "Wikidata", f"Wikidata entity {qid}", data_collected="metadata", reliability_0_100=70, notes="Structured public metadata; not performance stats.")
    return {"wikidata_qid": qid, "country_labels": "; ".join(countries) if countries else NA, "position_labels": "; ".join(positions) if positions else NA, "club_labels": "; ".join(clubs) if clubs else NA, "source_id": sid, "notes": "Wikidata metadata collected"}

def wikipedia_search_title(query):
    data = polite_get(f"{WIKIPEDIA_SEARCH}?action=query&list=search&srsearch={quote(query)}&format=json&srlimit=3", as_json=True, publisher="Wikipedia")
    res = data.get("query", {}).get("search", []) if data else []
    return res[0]["title"] if res else NA

def wikipedia_summary(query, entity_type="coach"):
    if not USE_WIKIPEDIA_SUMMARY:
        return {"summary": NA, "source_id": NA}
    title = wikipedia_search_title(query)
    if title == NA:
        return {"summary": NA, "source_id": NA}
    url = WIKIPEDIA_SUMMARY.format(title=quote(title.replace(" ", "_")))
    data = polite_get(url, as_json=True, publisher="Wikipedia")
    if not data:
        return {"summary": NA, "source_id": NA}
    sid = add_source("coach_stats", entity_type, query, url, "Wikipedia", title, data_collected="summary text", reliability_0_100=60, notes="Text only; not used to infer proxies.")
    return {"summary": data.get("extract", NA), "source_id": sid}

In [ ]:
# ────────────────────────────────────────────
# Montagem dos CSVs solicitados
# ────────────────────────────────────────────

def count_missing(row, cols):
    return sum(1 for c in cols if str(row.get(c, NA)) in ["", NA, "nan", "None", "<NA>"])

TOP5_LEAGUES = {"premier league","england premier league","la liga","laliga","spain la liga","serie a","italy serie a","bundesliga","germany bundesliga","ligue 1","france ligue 1"}

def is_top5_league_value(liga):
    if not isinstance(liga, str) or liga in ["", NA]:
        return NA
    return 1 if liga.strip().lower() in TOP5_LEAGUES else 0

player_stats_cols = [
    "player_id","selecao","jogador","posicao","clube","pais_do_clube","liga","idade","caps_selecao","gols_selecao",
    "minutes_last_365","matches_played_last_365","matches_started_last_365","club_goals_last_365","club_assists_last_365","xg_last_365","xa_last_365","shots_last_365","shots_on_target_last_365","key_passes_last_365","progressive_passes_last_365","progressive_carries_last_365","tackles_last_365","interceptions_last_365","blocks_last_365","clearances_last_365","aerial_duels_won_pct","pass_completion_pct","yellow_cards_last_365","red_cards_last_365","avg_player_rating_last_365",
    "gk_minutes_last_365","gk_matches_last_365","gk_save_pct_last_365","gk_clean_sheets_last_365","gk_goals_against_last_365","gk_xg_against_last_365","gk_goals_prevented_last_365","gk_penalty_save_pct_last_365",
    "injury_status","injury_type","expected_return_date","suspension_status","suspension_reason","fitness_status","expected_starter","starter_confidence_0_100",
    "market_value_eur","club_strength_0_100","league_strength_0_100","is_top5_league","played_elite_competition","elite_competition_minutes_last_365",
    "player_stats_source","injury_source","market_value_source","last_updated_at","research_confidence_0_100","missing_fields_count","notes"
]

player_rows = []
for _, base in tqdm(players_base.iterrows(), total=len(players_base), desc="Jogadores"):
    row = {c: NA for c in player_stats_cols}
    for c in ["player_id","selecao","jogador","posicao","clube","pais_do_clube","liga","idade","caps_selecao","gols_selecao"]:
        row[c] = base.get(c, NA) if str(base.get(c, "")) != "" else NA
    row["is_top5_league"] = is_top5_league_value(row["liga"])
    source_ids, notes = [], []
    if USE_WIKIDATA and row["jogador"] != NA:
        wd = wikidata_enrich_person_by_name(row["jogador"], "association football", "player")
        if wd:
            if row["posicao"] == NA and wd.get("position_labels", NA) != NA:
                row["posicao"] = wd["position_labels"]
            if row["clube"] == NA and wd.get("club_labels", NA) != NA:
                row["clube"] = wd["club_labels"]
            if wd.get("source_id", NA) != NA:
                source_ids.append(wd["source_id"])
            notes.append(wd.get("notes", "Wikidata checked"))
    row["expected_starter"] = "unknown"
    row["player_stats_source"] = ";".join(source_ids) if source_ids else NA
    row["last_updated_at"] = now_iso()
    row["research_confidence_0_100"] = 55 if source_ids else 20
    notes.append("Sem API key: performance, lesões, suspensões, rating, xG/xA e valor de mercado ficam NA se não vierem direto de fonte pública.")
    row["notes"] = " | ".join(notes)
    row["missing_fields_count"] = count_missing(row, player_stats_cols)
    player_rows.append(row)

player_stats_2025_26 = pd.DataFrame(player_rows, columns=player_stats_cols)
display(player_stats_2025_26.head())

In [ ]:
# Agregações por seleção e arquivos táticos/técnicos/lineups

def to_num(s):
    return pd.to_numeric(s.replace({NA: pd.NA, "": pd.NA}), errors="coerce")
def safe_sum(s):
    n = to_num(s); return NA if n.notna().sum()==0 else float(n.sum())
def safe_mean(s):
    n = to_num(s); return NA if n.notna().sum()==0 else float(n.mean())

team_agg_cols = ["selecao","squad_players_count","squad_avg_age","squad_total_caps","squad_total_international_goals","squad_minutes_last_365","squad_goals_club_last_365","squad_assists_club_last_365","squad_xg_last_365","squad_xa_last_365","squad_avg_player_rating_last_365","squad_avg_market_value_eur","squad_total_market_value_eur","squad_pct_top5_leagues","squad_avg_league_strength_0_100","squad_avg_club_strength_0_100","squad_elite_competition_minutes_last_365","top18_avg_rating_last_365","top18_minutes_last_365","top18_goals_club_last_365","top18_assists_club_last_365","top18_xg_last_365","top18_xa_last_365","top18_total_market_value_eur","top18_pct_top5_leagues","starting11_avg_age","starting11_total_caps","starting11_total_international_goals","starting11_minutes_last_365","starting11_goals_club_last_365","starting11_assists_club_last_365","starting11_xg_last_365","starting11_xa_last_365","starting11_avg_rating_last_365","starting11_total_market_value_eur","starting11_pct_top5_leagues","starting11_avg_league_strength_0_100","starting11_avg_club_strength_0_100","expected_starters_available","key_injuries_count","suspended_players_count","doubtful_players_count","rotation_risk_0_100","lineup_source_type","last_updated_at","missing_fields_count","notes"]

team_rows = []
for team in teams:
    g = player_stats_2025_26[player_stats_2025_26["selecao"] == team]
    row = {c: NA for c in team_agg_cols}
    row["selecao"] = team
    row["squad_players_count"] = int(len(g)) if len(g) else NA
    row["squad_avg_age"] = safe_mean(g["idade"]) if len(g) else NA
    row["squad_total_caps"] = safe_sum(g["caps_selecao"]) if len(g) else NA
    row["squad_total_international_goals"] = safe_sum(g["gols_selecao"]) if len(g) else NA
    row["squad_minutes_last_365"] = safe_sum(g["minutes_last_365"]) if len(g) else NA
    row["squad_goals_club_last_365"] = safe_sum(g["club_goals_last_365"]) if len(g) else NA
    row["squad_assists_club_last_365"] = safe_sum(g["club_assists_last_365"]) if len(g) else NA
    row["squad_xg_last_365"] = safe_sum(g["xg_last_365"]) if len(g) else NA
    row["squad_xa_last_365"] = safe_sum(g["xa_last_365"]) if len(g) else NA
    top5 = to_num(g["is_top5_league"]) if len(g) else pd.Series(dtype=float)
    row["squad_pct_top5_leagues"] = NA if top5.notna().sum()==0 else float((top5 == 1).mean() * 100)
    starters = g[g["expected_starter"].isin(["official","probable"])] if len(g) else g
    row["expected_starters_available"] = int(len(starters)) if len(starters) else NA
    row["lineup_source_type"] = "unknown"
    row["last_updated_at"] = now_iso()
    row["notes"] = "Top18 e starting11 não estimados. Sem proxy."
    row["missing_fields_count"] = count_missing(row, team_agg_cols)
    team_rows.append(row)
team_player_aggregates = pd.DataFrame(team_rows, columns=team_agg_cols)

team_tactical_cols = ["selecao","coach_name","main_formation","formation_last_match","formation_second_option","matches_analyzed_count","avg_possession_last_10","shots_for_last_10","shots_against_last_10","shots_on_target_for_last_10","shots_on_target_against_last_10","goals_for_last_10","goals_against_last_10","xg_for_last_10","xg_against_last_10","corners_for_last_10","corners_against_last_10","passes_attempted_last_10","pass_accuracy_pct_last_10","long_ball_pct_last_10","crosses_last_10","counter_attack_goals_last_10","set_piece_goals_for_last_10","set_piece_goals_against_last_10","ppda","high_turnovers_last_10","recoveries_final_third_last_10","pressing_intensity_0_100","defensive_block_height_0_100","directness_0_100","tempo_0_100","risk_tolerance_0_100","width_0_100","set_piece_attack_0_100","set_piece_defense_0_100","counter_attack_style_0_100","possession_style_0_100","tactical_data_type","tactical_source","last_updated_at","research_confidence_0_100","missing_fields_count","notes"]
team_tactical_real_stats = pd.DataFrame([{**{c: NA for c in team_tactical_cols}, "selecao": t, "tactical_data_type":"unavailable", "last_updated_at":now_iso(), "research_confidence_0_100":0, "notes":"Sem fonte pública sem key com métrica tática direta. Sem proxy."} for t in teams], columns=team_tactical_cols)
team_tactical_real_stats["missing_fields_count"] = team_tactical_real_stats.apply(lambda r: count_missing(r, team_tactical_cols), axis=1)

team_coach_cols = ["selecao","coach_name","coach_nationality","coach_start_date","coach_days_in_charge","coach_matches","coach_wins","coach_draws","coach_losses","coach_win_rate","coach_goals_for","coach_goals_against","coach_goals_for_avg","coach_goals_against_avg","coach_points_per_match","coach_major_tournaments","coach_main_formation","coach_style_description","coach_style_source","coach_stats_source","last_updated_at","research_confidence_0_100","missing_fields_count","notes"]

def wikidata_find_national_team_head_coach(team_name):
    if not USE_WIKIDATA:
        return {"coach_name": NA, "source_id": NA, "notes":"Wikidata disabled"}
    for q in [f"{team_name} national football team", f"{team_name} national soccer team", team_name]:
        cand = wikidata_search_entity(q, "en", 5)
        if cand:
            tqid = cand[0]["id"]
            ent = wikidata_get_entity(tqid)
            if ent:
                vals = wd_claim_values(ent, "P286")
                if vals:
                    cqid = wd_entity_id_from_value(vals[0])
                    cname = wd_label_from_entity_id(cqid) if cqid else NA
                    sid = add_source("coach_stats","team",team_name,WIKIDATA_ENTITY.format(qid=tqid),"Wikidata",f"Wikidata entity {tqid}",data_collected="head coach P286", reliability_0_100=65, notes="Verificar manualmente.")
                    return {"coach_name": cname, "source_id": sid, "notes":"Coach from Wikidata P286"}
    return {"coach_name": NA, "source_id": NA, "notes":"Coach not found in Wikidata"}

coach_rows = []
for team in tqdm(teams, desc="Técnicos"):
    row = {c: NA for c in team_coach_cols}
    row["selecao"] = team
    info = wikidata_find_national_team_head_coach(team)
    row["coach_name"] = info.get("coach_name", NA)
    sources = []
    if info.get("source_id", NA) != NA: sources.append(info["source_id"])
    if row["coach_name"] != NA:
        wd = wikidata_enrich_person_by_name(row["coach_name"], "football manager", "coach")
        if wd.get("country_labels", NA) != NA: row["coach_nationality"] = wd["country_labels"]
        if wd.get("source_id", NA) != NA: sources.append(wd["source_id"])
        wiki = wikipedia_summary(f"{row['coach_name']} football manager", "coach")
        if wiki.get("summary", NA) != NA:
            row["coach_style_description"] = wiki["summary"]
            row["coach_style_source"] = wiki.get("source_id", NA)
            sources.append(wiki.get("source_id", NA))
    row["coach_stats_source"] = ";".join([s for s in sources if s != NA]) if sources else NA
    row["last_updated_at"] = now_iso()
    row["research_confidence_0_100"] = 50 if sources else 0
    row["notes"] = info.get("notes","") + " | Histórico de jogos fica NA sem fonte pública estruturada sem key. Sem proxy."
    row["missing_fields_count"] = count_missing(row, team_coach_cols)
    coach_rows.append(row)
team_coach_real_stats = pd.DataFrame(coach_rows, columns=team_coach_cols)

lineups_cols = ["jogo","data","equipe","adversario","fase","grupo","expected_lineup_gk","expected_lineup_defenders","expected_lineup_midfielders","expected_lineup_forwards","expected_formation","expected_starters_count","expected_starters_available","key_injuries_count","key_injured_players","doubtful_players_count","doubtful_players","suspended_players_count","suspended_players","rotation_risk_0_100","lineup_source_type","lineup_source","injury_source","suspension_source","last_updated_at","research_confidence_0_100","missing_fields_count","notes"]
base = dataset_features if dataset_features is not None else matches_df
lineup_rows = []
if base is not None:
    tmp = base.copy(); tmp.columns = [normalize_colname(c) for c in tmp.columns]
    for c in ["jogo","data","fase","grupo","equipe1","equipe2"]:
        if c not in tmp.columns: tmp[c] = NA
    for _, m in tmp[["jogo","data","fase","grupo","equipe1","equipe2"]].drop_duplicates().iterrows():
        for team_col, opp_col in [("equipe1","equipe2"),("equipe2","equipe1")]:
            if str(m[team_col]).strip() and m[team_col] != NA:
                row = {c: NA for c in lineups_cols}
                row.update({"jogo":m["jogo"],"data":m["data"],"equipe":m[team_col],"adversario":m[opp_col],"fase":m["fase"],"grupo":m["grupo"],"lineup_source_type":"unknown","last_updated_at":now_iso(),"research_confidence_0_100":0,"notes":"Sem fonte pública sem key para lineups/lesões/suspensões. Sem proxy."})
                row["missing_fields_count"] = count_missing(row, lineups_cols)
                lineup_rows.append(row)
lineups_injuries_suspensions = pd.DataFrame(lineup_rows, columns=lineups_cols)
display(team_player_aggregates.head(), team_tactical_real_stats.head(), team_coach_real_stats.head(), lineups_injuries_suspensions.head())

In [ ]:
# ────────────────────────────────────────────
# Dataset final por jogo + dicionário
# ────────────────────────────────────────────

features = ["starting11_avg_rating_last_365","starting11_minutes_last_365","starting11_goals_club_last_365","starting11_assists_club_last_365","starting11_xg_last_365","starting11_xa_last_365","key_injuries_count","suspended_players_count","expected_starters_available","avg_possession_last_10","shots_for_last_10","shots_against_last_10","pressing_intensity_0_100","directness_0_100","defensive_block_height_0_100","coach_days_in_charge","coach_win_rate","coach_goals_for_avg","coach_goals_against_avg"]

def build_final_dataset():
    if dataset_features is not None:
        final = dataset_features.copy(); final.columns = [normalize_colname(c) for c in final.columns]
    elif matches_df is not None:
        final = matches_df.copy(); final.columns = [normalize_colname(c) for c in final.columns]
    else:
        final = pd.DataFrame(columns=["jogo","data","fase","grupo","equipe1","equipe2"])
    for c in ["equipe1","equipe2"]:
        if c not in final.columns: final[c] = NA
    agg = team_player_aggregates.set_index("selecao")
    tact = team_tactical_real_stats.set_index("selecao")
    coach = team_coach_real_stats.set_index("selecao")
    for side, col in [("team1","equipe1"),("team2","equipe2")]:
        for feat in features:
            values = []
            for team in final[col].astype(str):
                v = NA
                if feat in agg.columns and team in agg.index: v = agg.loc[team, feat]
                elif feat in tact.columns and team in tact.index: v = tact.loc[team, feat]
                elif feat in coach.columns and team in coach.index: v = coach.loc[team, feat]
                values.append(v)
            final[f"{side}_{feat}"] = values
    for feat in features:
        t1 = pd.to_numeric(final[f"team1_{feat}"].replace({NA: pd.NA, "": pd.NA}), errors="coerce")
        t2 = pd.to_numeric(final[f"team2_{feat}"].replace({NA: pd.NA, "": pd.NA}), errors="coerce")
        final[f"diff_{feat}"] = (t1 - t2).astype("Float64").astype(str).replace({"<NA>": NA})
        final[f"sum_{feat}"] = (t1 + t2).astype("Float64").astype(str).replace({"<NA>": NA})
    for c in ["style_similarity_0_100","team1_press_vs_team2_possession","team2_press_vs_team1_possession","team1_direct_attack_vs_team2_block","team2_direct_attack_vs_team1_block"]:
        final[c] = NA
    final["data_leakage_risk_players"] = "uncertain_check_source"
    final["data_leakage_risk_tactical"] = "uncertain_check_source"
    final["data_leakage_risk_lineups"] = "uncertain_check_source"
    final["data_leakage_risk_coach"] = "uncertain_check_source"
    return final

dataset_copa_2026_features_jogadores_tecnicos = build_final_dataset()

def make_dictionary_for_df(df, category, notes=""):
    rows = []
    for c in df.columns:
        rows.append({"column_name":c,"description":c.replace("_"," "),"data_type":"string","unit":"NA","allowed_values":"NA","source_category":category,"pre_match_allowed":"yes","data_leakage_risk":"uncertain_check_source","calculation_method":"direct_from_source","notes":notes})
    return rows

dict_rows = []
dict_rows += make_dictionary_for_df(player_stats_2025_26, "player_stats", "Sem proxy. NA sem fonte direta.")
dict_rows += make_dictionary_for_df(team_tactical_real_stats, "team_tactical", "Sem proxy. NA sem métrica direta.")
dict_rows += make_dictionary_for_df(team_coach_real_stats, "coach_stats", "Metadados via Wikidata/Wikipedia quando encontrados.")
dict_rows += make_dictionary_for_df(lineups_injuries_suspensions, "lineup", "NA sem fonte direta.")
dict_rows += make_dictionary_for_df(dataset_copa_2026_features_jogadores_tecnicos, "final_dataset", "Diff/sum por aritmética direta.")
data_dictionary_jogadores_tecnicos = pd.DataFrame(dict_rows).drop_duplicates(subset=["column_name","source_category"])
display(dataset_copa_2026_features_jogadores_tecnicos.head())

In [ ]:
# ────────────────────────────────────────────
# Salvamento incremental no repositório
# ────────────────────────────────────────────

def clean_for_csv(df):
    return df.copy().replace({pd.NA: NA, None: NA, "nan": NA, "NaN": NA}).fillna(NA)

def is_missing_value(x):
    return pd.isna(x) or str(x).strip() in ["", "NA", "nan", "NaN", "None", "<NA>"]

def merge_incremental(existing_df, new_df, key_cols, update_only_missing=True):
    if existing_df is None or existing_df.empty:
        return new_df.copy()
    old, new = existing_df.copy(), new_df.copy()
    for c in new.columns:
        if c not in old.columns: old[c] = NA
    for c in old.columns:
        if c not in new.columns: new[c] = NA
    cols = list(dict.fromkeys(list(old.columns) + list(new.columns)))
    old, new = old[cols], new[cols]
    old["_key"] = old[key_cols].astype(str).agg("||".join, axis=1)
    new["_key"] = new[key_cols].astype(str).agg("||".join, axis=1)
    rows = old.to_dict("records")
    index = {r["_key"]: i for i, r in enumerate(rows)}
    for _, nr in new.iterrows():
        d = nr.to_dict()
        k = d["_key"]
        if k not in index:
            rows.append(d); continue
        r = rows[index[k]]
        for c in cols:
            if c == "_key": continue
            if update_only_missing:
                if is_missing_value(r.get(c, NA)) and not is_missing_value(d.get(c, NA)):
                    r[c] = d[c]
            else:
                if not is_missing_value(d.get(c, NA)):
                    r[c] = d[c]
    return pd.DataFrame(rows).drop(columns=["_key"], errors="ignore")[cols]

def save_csv_incremental(df, filename, key_cols):
    path = OUTPUT_DIR / filename
    new_df = clean_for_csv(df)
    if path.exists():
        old_df = pd.read_csv(path, dtype=str, keep_default_na=False, na_values=[])
        merged = merge_incremental(old_df, new_df, key_cols, UPDATE_ONLY_MISSING)
    else:
        merged = new_df
    merged.to_csv(path, index=False, encoding="utf-8", na_rep=NA)
    return path, merged

sources_used_jogadores_tecnicos = pd.DataFrame(sources_rows, columns=["source_id","category","entity_type","entity_name","url","publisher","title","date_published","date_accessed","data_collected","reliability_0_100","notes"])
if sources_used_jogadores_tecnicos.empty:
    sources_used_jogadores_tecnicos = pd.DataFrame([{"source_id":"NA","category":"NA","entity_type":"NA","entity_name":"NA","url":"NA","publisher":"NA","title":"NA","date_published":"NA","date_accessed":now_iso(),"data_collected":"No source accessed","reliability_0_100":0,"notes":"No no-key source found."}])
request_log_df = pd.DataFrame(request_log) if request_log else pd.DataFrame([{"timestamp": now_iso(), "status": "no_requests"}])

saved = {}
saved["player_stats_2025_26.csv"] = save_csv_incremental(player_stats_2025_26, "player_stats_2025_26.csv", ["player_id","selecao","jogador"])
saved["team_tactical_real_stats.csv"] = save_csv_incremental(team_tactical_real_stats, "team_tactical_real_stats.csv", ["selecao"])
saved["team_coach_real_stats.csv"] = save_csv_incremental(team_coach_real_stats, "team_coach_real_stats.csv", ["selecao","coach_name"])
saved["lineups_injuries_suspensions.csv"] = save_csv_incremental(lineups_injuries_suspensions, "lineups_injuries_suspensions.csv", ["jogo","equipe","adversario"])
saved["dataset_copa_2026_features_jogadores_tecnicos.csv"] = save_csv_incremental(dataset_copa_2026_features_jogadores_tecnicos, "dataset_copa_2026_features_jogadores_tecnicos.csv", ["jogo"] if "jogo" in dataset_copa_2026_features_jogadores_tecnicos.columns else ["equipe1","equipe2"])
saved["data_dictionary_jogadores_tecnicos.csv"] = save_csv_incremental(data_dictionary_jogadores_tecnicos, "data_dictionary_jogadores_tecnicos.csv", ["column_name","source_category"])
saved["sources_used_jogadores_tecnicos.csv"] = save_csv_incremental(sources_used_jogadores_tecnicos, "sources_used_jogadores_tecnicos.csv", ["category","entity_type","entity_name","url"])
request_log_path, request_log_final = save_csv_incremental(request_log_df, "request_log.csv", ["timestamp","status"])

report = []
report.append("# Relatório incremental — jogadores, técnicos e seleções")
report.append("")
report.append(f"Coleta gerada em: {now_iso()}")
report.append(f"Repositório: {REPO_ROOT}")
report.append(f"Saída: {OUTPUT_DIR}")
report.append(f"UPDATE_ONLY_MISSING: {UPDATE_ONLY_MISSING}")
report.append("")
report.append("## Entradas")
for k, v in required_candidates.items():
    report.append(f"- {k}: {v if v else 'NÃO ENCONTRADO'}")
report.append("")
report.append("## Regra")
report.append("Sem proxy, sem estimativa, sem API key. Campos sem dado direto ficam NA.")
report.append("")
report.append("## Arquivos salvos")
for filename, (path, df) in saved.items():
    report.append(f"- {filename}: {df.shape[0]} linhas x {df.shape[1]} colunas")
report_path = OUTPUT_DIR / "relatorio_pesquisa_jogadores_tecnicos.md"
report_path.write_text("\n".join(report), encoding="utf-8")

zip_final_path = OUTPUT_DIR / "pacote_jogadores_tecnicos_copa_2026.zip"
with zipfile.ZipFile(zip_final_path, "w", zipfile.ZIP_DEFLATED) as z:
    for filename, (path, df) in saved.items():
        z.write(path, arcname=filename)
    z.write(report_path, arcname=report_path.name)
    z.write(request_log_path, arcname=request_log_path.name)

print("Salvo em:", OUTPUT_DIR)
print("ZIP final:", zip_final_path)
for filename, (path, df) in saved.items():
    print("-", filename, df.shape)

## Como atualizar depois

Para atualizar com novos dados:

1. Rode o notebook novamente no mesmo repositório.
2. Ele vai ler os CSVs já existentes em `data/database/enriched_jogadores_tecnicos/`.
3. Quando encontrar dados novos, preenche somente campos `NA` ou vazios.
4. Dados já preenchidos ficam preservados.

Para sobrescrever valores antigos com dados novos não-NA, altere:

```python
UPDATE_ONLY_MISSING = False
```

O notebook também mantém cache em `.copa_2026_request_cache`, evitando repetir requisições já feitas.